# Azayaka チュートリアル `ジオコード`編

![img](https://www.eorc.jaxa.jp/ALOS/jp/dataset/images/Noto_obsArea_20240110_rev2.png)
**© JAXA**

JAXA より提供されている2024年1月能登半島地震の時のa ALOS-2 PLASAR-2 の観測実績のオープンデータ

https://www.eorc.jaxa.jp/ALOS/jp/dataset/open_and_free/palsar2_l11_l22_j.htm


公式よりサンプルデータをダウンロードします。

In [1]:
import os

if True:
  # JAXA ALOS-2 PALSAR-2 L2.1 (GeoTIFF) のサンプルデータURL
  zip_url = "https://www.eorc.jaxa.jp/ALOS/jp/dataset/open_and_free/202401_noto_eq/L11_ALOS2515060720-231206.zip"
  output_zip = "palsar2_data.zip"

  print(f"{zip_url} からダウンロードを開始します...")
  # wgetを使用してダウンロードを実行
  !wget -q --show-progress -O {output_zip} {zip_url}

  if os.path.exists(output_zip):
      print(f"\nダウンロード完了: {output_zip}")

https://www.eorc.jaxa.jp/ALOS/jp/dataset/open_and_free/202401_noto_eq/L11_ALOS2515060720-231206.zip からダウンロードを開始します...
palsar2_data.zip    100%[===================>]   7.00G  1.18MB/s    in 1h 40m  

ダウンロード完了: palsar2_data.zip


ZIPファイルを展開します。

In [2]:
if True:
  # 展開先のディレクトリ名
  extract_dir = "palsar2_extracted"

  # zipファイルの存在を確認して展開
  if os.path.exists("palsar2_data.zip"):
      print(f"{extract_dir} に展開しています...")
      !mkdir -p {extract_dir}
      !unzip -o palsar2_data.zip -d {extract_dir}
      print(f"展開が完了しました。")

palsar2_extracted に展開しています...
Archive:  palsar2_data.zip
   creating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/
 extracting: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/BRS-HH-ALOS2515060720-231206-UBSR1.1__A.jpg  
  inflating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/IMG-HH-ALOS2515060720-231206-UBSR1.1__A  
  inflating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/LED-ALOS2515060720-231206-UBSR1.1__A  
  inflating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/summary.txt  
  inflating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/TRL-ALOS2515060720-231206-UBSR1.1__A  
  inflating: palsar2_extracted/0000541307_004001_ALOS2515060720-231206/VOL-ALOS2515060720-231206-UBSR1.1__A  
展開が完了しました。


# Azayaka

![img](https://raw.githubusercontent.com/syu-tan/azayaka/main/doc/figure/001_project_icon.png)

`azayaka` ライブラリの環境構築とインストールを行います。

In [6]:
!pip install -qqq azayaka

インポートをしてバージョンの確認をします。

In [9]:
import numpy as np

from azayaka.fileformat import CEOS_PALSAR2_L11_SLC
from azayaka.geocode import Geocode


 ____    _    ____        _                                    _            
/ ___|  / \  |  _ \      / \     ____   __ _   _   _    __ _  | | __   __ _ 
\___ \ / _ \ | |_) |    / _ \   |_  /  / _` | | | | |  / _` | | |/ /  / _` |
 ___) / ___ \|  _ <    / ___ \   / /  | (_| | | |_| | | (_| | |   <  | (_| |
|____/_/   \_\_| \_\  /_/   \_\ /___|  \__,_|  \__, |  \__,_| |_|\_\  \__,_|
                                               |___/   
 Version 0.1.1


もし、Google Drive にデータがある場合は、パスを変更して読み込むことができます。

クラウドダウンロードではとても時間がかかります。Google Drive のマウントをお勧めします。

In [7]:
# from google.colab import drive
# drive.mount('/content/drive')

# PATH_SAR = "/content/drive/MyDrive/azayaka/sample/ALOS2515060720-231206" # your google drive
PATH_SAR = "./palsar2_extracted/0000541307_004001_ALOS2515060720-231206/" # download

ジオコードに使用する `DEM` (数値標高モデル) を取得します。

In [8]:
!wget -O dem_noto_alos2.tif https://github.com/syu-tan/azayaka/raw/main/example/dem/dem_noto_alos2.tif

--2026-04-08 05:45:52--  https://github.com/syu-tan/azayaka/raw/main/example/dem/dem_noto_alos2.tif
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/syu-tan/azayaka/main/example/dem/dem_noto_alos2.tif [following]
--2026-04-08 05:45:52--  https://raw.githubusercontent.com/syu-tan/azayaka/main/example/dem/dem_noto_alos2.tif
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 658446 (643K) [image/tiff]
Saving to: ‘dem_noto_alos2.tif’

dem_noto_alos2.tif  100%[===================>] 643.01K  --.-KB/s    in 0.03s   

2026-04-08 05:45:53 (19.9 MB/s) - ‘dem_noto_alos2.tif’ saved [658446/658446]



入力データと出力データの可視化をしておきます。

In [10]:
path_ceos = os.path.join(PATH_SAR)
path_dem = os.path.join("dem_noto_alos2.tif")

# Output GeoTIFFs
out_dir = os.path.join("output/geocode")
os.makedirs(out_dir, exist_ok=True)
out_intensity = os.path.join(out_dir, "geocoded_intensity.tif")
out_phase = os.path.join(out_dir, "geocoded_phase.tif")
out_kml = os.path.join(out_dir, "geocoded_scene_footprint.kml")
out_intensity_bounds = os.path.join(out_dir, "geocoded_intensity_bounds.tif")
output_geometory_json = os.path.join(out_dir, "geocoded_geometry.json")
print(f"Input: {path_ceos}  Output directory: {out_dir}")

Input: ./palsar2_extracted/0000541307_004001_ALOS2515060720-231206/  Output directory: output/geocode


# ジオコード

まずは、CEOSフォーマットの読み込みとジオメトリを算出します。

In [11]:
# ---- Load CEOS PALSAR-2 L1.1 ----
ceos = CEOS_PALSAR2_L11_SLC(
    PATH_CEOS_FOLDER=path_ceos,
    POLARIMETORY="HH",
    ORBIT_NAME="A",
)
ceos.set_geometory(plot=True, output_json_path=output_geometory_json)

IMG: B9-12 SAR イメージファイルディスクリプタレコード長 -> 720
IMG: bits/sample   -> 32
IMG: samples/group -> 2
IMG: bytes/group   -> 8
IMG: B277-280 PREFIX DATA bytes/record -> 544
IMG: B181-186 SARデータレコード数 -> 37309
IMG: B187-192 SARデータレコード長 -> 216736
========== IMG Header (prefix=544) ==========
IMG: B49-50 SARチャンネルID -> 1
IMG: B9-12 レコード長（シグナルレコード長） -> 216736
IMG: B13-16 SAR画像データライン番号 -> 1
IMG: B17-20 SAR画像データレコードインデックス -> 1
IMG: B25-28 実際のデータピクセル数 -> 27024
IMG: B29-32 実際の右詰めのピクセル数 -> 0
IMG: B57-60 PRF [mHz] -> 3730832
IMG: B67-68 チャープ形式指定者 -> 0
IMG: B69-72 チャープ長(パルス幅) nsec -> 53287
IMG: B73-76 チャープ定数係数 Hz -> 0
IMG: B77-80 チャープ一次係数 Hz/μsec -> 1490044
IMG: B81-84 チャープ二次係数 Hz/μsec^2 -> 0
IMG: B93-96 受信機ゲイン dB -> 86
IMG: B117-120 最初のデータまでのスラントレンジ [m] -> 816697
IMG: B121-124 SAMPLE DELAY [nsec] -> 5448418
  -> TIME_GATE_DELAY [sec] = 0
Num of Range (pixels): 27024 Num of Azimuth Line: 37309


Read SLC (ALOS-2 L1.1):   0%|          | 94/37309 [00:00<00:39, 939.66it/s]

========== Start Time ==========
IMG: B37-40 センサー取得年 -> 2023
IMG: B41-44 センサー取得日(通算) -> 340
IMG: B45-48 センサー取得ミリ秒 -> 53508859
IMG Prefix: SARチャンネルID -> 1
IMG Prefix: SARチャンネルコード -> 0
IMG Prefix: Tx偏波コード -> 0
IMG Prefix: Rx偏波コード -> 0


Read SLC (ALOS-2 L1.1): 100%|██████████| 37309/37309 [00:34<00:00, 1082.75it/s]


========== End Time ==========
IMG: B37-40 センサー取得年 -> 2023
IMG: B41-44 センサー取得日(通算) -> 340
IMG: B45-48 センサー取得ミリ秒 -> 53518859
LED: B9-12 リーダファイルディスクリプタレコード長 -> 720
LED: データセットサマリレコード数 -> 1
LED: データセットサマリレコード長 -> 4096
LED: 地図投影データレコード数 -> 0
LED: 地図投影データレコード長 -> 0
LED: プラットフォーム位置データレコード長 -> 4680
LED: 姿勢データレコード長 -> 16384
LED: データセットサマリレコード長(B4) -> 4096
LED: シーンID -> ALOS2515060720-231206
LED: シーンセンター時刻 -> 20231206145153859
LED: 楕円体モデル -> GRS80
LED: 楕円体の半長径(Km) -> 6378.137
LED: 楕円体の短半径(Km) -> 6356.7523141
LED: 地球の質量(10^24 Kg) -> 5.974
LED: J2 -> 0.1082629
LED: J3 -> -2.54e-05
LED: J4 -> -1.62e-05
LED: SARチャネル数 -> 4
LED: センサプラットフォーム名 -> ALOS2
LED: 波長 λ [m] -> 0.2384035
LED: Motion compensation indicator -> 00
LED: レンジパルスコード -> LINEAR FM CHIRP
LED: レンジパルス振幅係数1 -> 0.0
LED: レンジパルス振幅係数2 -> -1490052400000.0
LED: レンジパルス振幅係数3 -> 0.0
LED: レンジパルス振幅係数4 -> 0.0
LED: レンジパルス振幅係数5 -> 0.0
LED: サンプリング周波数(MHz) -> 104.7915957
LED: レンジゲート(μsec) -> 91.6104
LED: レンジパルス幅(μsec) -> 53.286716
LED: 量子化記述子 -> UNIFORM I,

レンジドップラー方程式に基づいて、ジオコードを実施します。

In [ ]:
# ---- Geocode ----
geocoder = Geocode(
    sar=ceos,
    dem_path=path_dem,
    buffer_sample=0,
)

out = geocoder.geocode(
    signal=ceos.signal,
    output_intensity_path=out_intensity,
    output_phase_path=out_phase,
    register=True,
)

print(f"Intensity GeoTIFF: {out_intensity}")
print(f"Phase GeoTIFF: {out_phase}")
print(f"Scene KML: {out_kml}")
print(f"Intensity Bounds GeoTIFF: {out_intensity_bounds}")
print(f"Geometry JSON: {output_geometory_json}")

Mapping DEM to Radar: 100%|██████████| 473/473 [00:00<00:00, 1810.36it/s]


## 結果の確認

ジオコードした GeoTIFF のダウンロード

In [ ]:
from google.colab import files

if os.path.exists(out_intensity):
    print(f'Downloading {out_intensity}...')
    files.download(out_intensity)

## 結果の可視化

In [ ]:
!pip install -qqq leafmap xarray localtileserver --ignore-installed blinker

In [ ]:
import leafmap
import rasterio

# データの統計量を計算してストレッチ範囲（2%~98%）を特定
with rasterio.open(out_intensity) as src:
    data = src.read(1)
    # 0や無効値を除外して計算
    valid_data = data[data > 0]
    vmin, vmax = np.percentile(valid_data, [2, 98])

print(f"Stretch range: {vmin} to {vmax}")

m = leafmap.Map(basemap='OpenStreetMap')
# 計算した vmin, vmax を適用して描画
m.add_raster(out_intensity, layer_name='SAR Intensity (Stretched)', cmap='gray', vmin=vmin, vmax=vmax)
m

![img](../../doc/figure/007_tutorial_geocode.png)

**© JAXA**